In [ ]:
import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
from torchvision.models import vgg16
import matplotlib.pyplot as plt
import time
import numpy as np

In [ ]:
class Residual(nn.Module):
    def __init__(self, num_channels):
        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(num_channels, num_channels, kernel_size=3, stride=1, padding=1),
            nn.GroupNorm(32, num_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(num_channels, num_channels, kernel_size=3, stride=1, padding=1),
            nn.GroupNorm(32, num_channels)
        )

    def forward(self, x):
        return x + self.block(x)

class Encoder(nn.Module):
    def __init__(self, hidden_dim=128, embedding_dim=128, num_channels=3):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, hidden_dim // 2, kernel_size=4, stride=2, padding=1),                      # 512 -> 256
            nn.GroupNorm(32, hidden_dim // 2),
            nn.ReLU(),

            nn.Conv2d(hidden_dim // 2, hidden_dim, kernel_size=4, stride=2, padding=1),             # 256 -> 128
            nn.GroupNorm(32, hidden_dim),
            nn.ReLU(),

            nn.Conv2d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1),              # 128 -> 64
            nn.GroupNorm(32, hidden_dim * 2),
            nn.ReLU(),

            nn.Conv2d(hidden_dim * 2, hidden_dim * 4, kernel_size=4, stride=2, padding=1),          # 64 -> 32
            nn.GroupNorm(32, hidden_dim * 4),
            nn.ReLU(),

            nn.Conv2d(hidden_dim * 4, embedding_dim, kernel_size=3, stride=1, padding=1),          # 32 -> 32
            nn.GroupNorm(32, embedding_dim)
        )
        self.residual = Residual(embedding_dim)

    def forward(self, x):
        x = self.conv(x)
        return self.residual(x)

class Decoder(nn.Module):
    def __init__(self, embedding_dim=128, hidden_dim=128, num_channels=3):
        super().__init__()

        self.residual = Residual(embedding_dim)

        self.conv = nn.Sequential(
            nn.ConvTranspose2d(embedding_dim, hidden_dim * 4, kernel_size=3, stride=1, padding=1),             # 32 -> 32
            nn.GroupNorm(32, hidden_dim * 4),
            nn.ReLU(),

            nn.ConvTranspose2d(hidden_dim * 4, hidden_dim * 2, kernel_size=4, stride=2, padding=1),             # 32 -> 64
            nn.GroupNorm(32, hidden_dim * 2),
            nn.ReLU(),

            nn.ConvTranspose2d(hidden_dim * 2, hidden_dim, kernel_size=4, stride=2, padding=1),              # 64 -> 128
            nn.GroupNorm(32, hidden_dim),
            nn.ReLU(),

            nn.ConvTranspose2d(hidden_dim, hidden_dim // 2, kernel_size=4, stride=2, padding=1),          # 128 -> 256
            nn.GroupNorm(32, hidden_dim // 2),
            nn.ReLU(),

            nn.ConvTranspose2d(hidden_dim // 2, num_channels, kernel_size=4, stride=2, padding=1)           # 256 -> 512
        )
    def forward(self, x):
        x = self.residual(x)
        x = self.conv(x)
        return torch.tanh(x)

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=512, embedding_dim=128, commitment_cost=0.25):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost

        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-0.1, 0.1)

    def forward(self, z):
        z_flattened = z.permute(0, 2, 3, 1).contiguous()
        z_flattened = z_flattened.view(-1, self.embedding_dim)

        distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                     + torch.sum(self.embeddings.weight**2, dim=1)
                     - 2 * torch.matmul(z_flattened, self.embeddings.weight.t()))

        encoding_indices = torch.argmin(distances, dim=1)
        encodings = F.one_hot(encoding_indices, self.num_embeddings).float()

        quantized = torch.matmul(encodings, self.embeddings.weight)
        quantized = quantized.view(z.shape[0], z.shape[2], z.shape[3], self.embedding_dim)
        quantized = quantized.permute(0, 3, 1, 2).contiguous()

        e_latent_loss = F.mse_loss(quantized.detach(), z)
        q_latent_loss = F.mse_loss(quantized, z.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss

        quantized = z + (quantized - z).detach()

        avg_probs = torch.mean(encodings, dim=0)

        return quantized, loss

class VQ_VAE(nn.Module):
    def __init__(self, num_embeddings=512, embedding_dim=128, hidden_dim=128, commitment_cost=0.25):
        super().__init__()
        self.encoder = Encoder(hidden_dim, embedding_dim)
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.decoder = Decoder(embedding_dim, hidden_dim)

    def forward(self, x):
        z = self.encoder(x)
        quantized, vq_loss, perplexity, active_codes = self.vq(z)
        x_recon = self.decoder(quantized)
        return x_recon, vq_loss
